# Train Simhastha Suraksha on Kaggle

Run top to bottom on a Kaggle notebook with **GPU on** (Settings, Accelerator) and the ShanghaiTech crowd-counting dataset attached (Add data).
Check the dataset has `train_data/images` and `train_data/ground_truth` (.mat files). Sessions end, so download `runs/` and `models/` before closing.

## 1. Get the code
If the repo is private, create a GitHub token with read access and put it in Kaggle Secrets as `GH_TOKEN`. Never paste a token into a cell.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")  # remove these two lines if the repo is public
!git clone https://{token}@github.com/sanket1035/simhastha-suraksha.git
%cd simhastha-suraksha
!pip install -q -r requirements-train.txt

## 2. Find the dataset and build density maps

In [ ]:
import glob, os
# Finds the Part A folder (the one that contains train_data) inside the attached dataset.
cands = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train_data", recursive=True))
print("Found:", cands)
assert cands, "No train_data folder found. Attach the ShanghaiTech crowd-counting dataset (Add data)."
part_a = [c for c in cands if "part_a" in c.lower() or "part a" in c.lower() or "parta" in c.lower()]
DATA_ROOT = (part_a or cands)[0]
print("Using DATA_ROOT =", DATA_ROOT)
!ls {DATA_ROOT}

In [ ]:
!python -m src.data_prep --root {DATA_ROOT} --part A --out data/processed
# Check the output above: every line should say "N images". Any WARNING means the annotations did not match.

## 3. Baselines (MLP and CNN)
Each run writes `runs/<name>/log.csv` and `best.pth`. Keep epochs small first to check everything works.

In [ ]:
IDX = "data/processed/part_A/train_index.csv"
!python -m src.train --model mlp --train-index {IDX} --out runs/mlp --epochs 30 --lr 1e-3
!python -m src.train --model cnn --train-index {IDX} --out runs/cnn --epochs 30 --lr 1e-3
# Regularization and activations (Experiments 3 and 4), examples:
!python -m src.train --model cnn --train-index {IDX} --out runs/cnn_dropout --epochs 30 --lr 1e-3 --dropout 0.3
!python -m src.train --model cnn --train-index {IDX} --out runs/cnn_l2 --epochs 30 --lr 1e-3 --weight-decay 1e-4
!python -m src.train --model cnn --train-index {IDX} --out runs/cnn_sigmoid --epochs 30 --lr 1e-3 --activation sigmoid

## 4. CSRNet (smoke test first, then the real run)

In [ ]:
!python -m src.train --model csrnet --train-index {IDX} --out runs/csrnet_smoke --epochs 1 --max-train-images 10 --amp

In [ ]:
!python -m src.train --model csrnet --train-index {IDX} --out runs/csrnet_A --epochs 100 --lr 1e-5 --optimizer adam --amp
# Freeze vs fine-tune (Experiment 7):
# !python -m src.train --model csrnet --train-index {IDX} --out runs/csrnet_frozen --epochs 50 --lr 1e-5 --amp --freeze-frontend

## 5. Evaluate on the test split and export

In [ ]:
TEST = "data/processed/part_A/test_index.csv"
!python -m src.evaluate --checkpoint runs/csrnet_A/best.pth --test-index {TEST} --out runs/csrnet_A/eval --save-worst 10
!python -m src.export_onnx --checkpoint runs/csrnet_A/best.pth --out models/csrnet.onnx

## 6. Download results
Zip `runs/` and `models/csrnet.onnx`, download them, and attach the ONNX file to a GitHub Release (not to git).

In [ ]:
!zip -rq results.zip runs models/csrnet.onnx